# SymPy for Symbolic Engineering

## The Core Question

**How do you derive equations symbolically before coding them numerically?**

Every engineering code starts with **equations**: the beam deflection equation, the heat equation, the Navier-Stokes equations. You can either:
1. **Derive the equations by hand** (slow, error-prone), or
2. **Derive the equations symbolically** (fast, accurate, reproducible).

SymPy is the **standard library** for symbolic mathematics in Python. It lets you:
- **Define symbols** (e.g., `x`, `y`, `E`, `I`)
- **Differentiate** and **integrate** symbolically
- **Solve** equations (algebraic, differential)
- **Simplify** complex expressions
- **Generate numerical code** from symbolic expressions (`lambdify`)

This is the **exact workflow** used in every engineering code: derive the equations symbolically, then generate the numerical code.

## Why this library?

- **It is the standard.** Every symbolic computation in Python uses SymPy. Learning it is non-negotiable.
- **It is accurate.** SymPy uses exact arithmetic (fractions, radicals), so there's no floating-point error.
- **It is flexible.** You can define your own symbols, functions, and equations. You can solve algebraic equations, differential equations, and systems of equations.

## The Workflow

```
Engineering problem (beam, heat, fluid)
        |
        v
Define the symbols (x, y, E, I, ...)
        |
        v
Write the governing equation (symbolically)
        |
        v
Solve the equation (symbolically)
        |
        v
Simplify the solution
        |
        v
Generate numerical code (lambdify)
        |
        v
Evaluate + visualize (NumPy, Matplotlib)
```

## Setup


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Derive the Beam Deflection Equation (Euler-Bernoulli)

### The concept

The **Euler-Bernoulli beam equation** relates the bending moment `M(x)` to the deflection `w(x)`:

```
E * I * d²w/dx² = M(x)
```

where `E` is the Young's modulus, `I` is the second moment of area, and `M(x)` is the bending moment. For a **simply supported beam** with a **point load** `P` at the center, the bending moment is:

```
M(x) = P * x / 2  (for 0 <= x <= L/2)
     = P * (L - x) / 2  (for L/2 <= x <= L)
```

We want to **derive the deflection equation** `w(x)` symbolically.

### Engineering case: a simply supported beam with a point load

A beam of length `L`, with a point load `P` at the center. We want to derive the deflection equation `w(x)` and evaluate it at the center (where the deflection is maximum).


In [ ]:
# Define the symbols
x, L, P, E, I = sp.symbols('x L P E I', positive=True)

# Bending moment for a simply supported beam with a point load at the center
# M(x) = P * x / 2  (for 0 <= x <= L/2)
# M(x) = P * (L - x) / 2  (for L/2 <= x <= L)
# Use the left-half expression (valid for 0 <= x <= L/2)
M = P * x / 2

# Euler-Bernoulli beam equation: E * I * w'' = M
# w'' = M / (E * I)
# w' = integral(M / (E * I), x) + C1
# w = integral(w', x) + C2

C1, C2 = sp.symbols('C1 C2')
w_pp = M / (E * I)
w_p = sp.integrate(w_pp, x) + C1
w_expr = sp.integrate(w_p, x) + C2

# Apply boundary conditions:
# 1. w(0) = 0 (simply supported at x = 0)
# 2. w'(L/2) = 0 (symmetry: the slope is zero at the center)
bc1 = w_expr.subs(x, 0)  # w(0) = 0
bc2 = w_p.subs(x, L / 2)  # w'(L/2) = 0

# Solve for C1 and C2
sol = sp.solve([bc1, bc2], [C1, C2])
w_final = w_expr.subs(sol)

# Simplify the deflection equation
w_final = sp.simplify(w_final)
print("Deflection equation (0 <= x <= L/2):")
print(w_final)

# Evaluate the deflection at the center (x = L/2)
w_center = w_final.subs(x, L / 2)
w_center = sp.simplify(w_center)
print(f"\nDeflection at the center (x = L/2):")
print(w_center)

# Analytical solution: w_center = P * L³ / (48 * E * I)
# (The negative sign indicates the deflection is downward)
w_analytical = -P * L**3 / (48 * E * I)
print(f"\nAnalytical solution: -P * L³ / (48 * E * I) (negative = downward)")
print(f"Match: {sp.simplify(w_center - w_analytical) == 0}")

### What does this mean?

The deflection equation is:

```
w(x) = P * x * (4*x² - 3*L²) / (48 * E * I)  (for 0 <= x <= L/2)
```

The deflection at the center is:

```
w(L/2) = -P * L³ / (48 * E * I)
```

The **negative sign** indicates the deflection is **downward** (the beam bends down under the load). The **magnitude** `P * L³ / (48 * E * I)` matches the **analytical solution** exactly. The symbolic derivation is **fast, accurate, and reproducible** — you can re-run the code and get the same result every time.

**Engineering takeaway:** This is the **exact workflow** you would use to derive a beam deflection equation. In a real workflow, you would:
1. Define the symbols (`x`, `L`, `P`, `E`, `I`).
2. Write the governing equation (`E * I * w'' = M`).
3. Integrate twice (`sp.integrate`).
4. Apply the boundary conditions (`w(0) = 0`, `w'(L/2) = 0` for symmetry).
5. Solve for the constants (`sp.solve`).
6. Simplify the solution (`sp.simplify`).
7. Evaluate the solution at the point of interest (`w(L/2)`).
8. Interpret the sign (negative = downward deflection).


## 2. Symbolic Differentiation of a Shape Function

### The concept

A **shape function** (or **basis function**) is a function that interpolates the solution at a set of points. In the **Finite Element Method (FEM)**, shape functions are used to approximate the solution within each element.

For a **linear element** (2 nodes), the shape functions are:

```
N1(x) = (x2 - x) / (x2 - x1)
N2(x) = (x - x1) / (x2 - x1)
```

where `x1` and `x2` are the node coordinates. We want to **differentiate the shape functions** symbolically to get the **strain** (the derivative of the displacement).

### Engineering case: a linear FEM element

A linear element with nodes at `x1 = 0` and `x2 = L`. We want to differentiate the shape functions to get the strain.


In [ ]:
# Define the symbols
x, x1, x2 = sp.symbols('x x1 x2')

# Shape functions for a linear element
N1 = (x2 - x) / (x2 - x1)
N2 = (x - x1) / (x2 - x1)

# Differentiate the shape functions
dN1_dx = sp.diff(N1, x)
dN2_dx = sp.diff(N2, x)

print("Shape functions:")
print(f"N1(x) = {N1}")
print(f"N2(x) = {N2}")
print(f"\nDerivatives:")
print(f"dN1/dx = {dN1_dx}")
print(f"dN2/dx = {dN2_dx}")

# Simplify the derivatives
dN1_dx_simplified = sp.simplify(dN1_dx)
dN2_dx_simplified = sp.simplify(dN2_dx)
print(f"\nSimplified derivatives:")
print(f"dN1/dx = {dN1_dx_simplified}")
print(f"dN2/dx = {dN2_dx_simplified}")

# Evaluate the derivatives at specific node coordinates
x1_val, x2_val = 0, 1.0  # nodes at x = 0 and x = 1
dN1_dx_eval = dN1_dx_simplified.subs({x1: x1_val, x2: x2_val})
dN2_dx_eval = dN2_dx_simplified.subs({x1: x1_val, x2: x2_val})
print(f"\nDerivatives at x1 = 0, x2 = 1:")
print(f"dN1/dx = {dN1_dx_eval}")
print(f"dN2/dx = {dN2_dx_eval}")

# Plot the shape functions and their derivatives
x_vals = np.linspace(0, 1, 100)
N1_vals = [float(N1.subs({x: xi, x1: x1_val, x2: x2_val})) for xi in x_vals]
N2_vals = [float(N2.subs({x: xi, x1: x1_val, x2: x2_val})) for xi in x_vals]
dN1_dx_vals = [float(dN1_dx_simplified.subs({x: xi, x1: x1_val, x2: x2_val})) for xi in x_vals]
dN2_dx_vals = [float(dN2_dx_simplified.subs({x: xi, x1: x1_val, x2: x2_val})) for xi in x_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(x_vals, N1_vals, 'b-', linewidth=2, label='N1(x)')
ax1.plot(x_vals, N2_vals, 'r-', linewidth=2, label='N2(x)')
ax1.set_xlabel('x')
ax1.set_ylabel('N(x)')
ax1.set_title('Shape functions')
ax1.legend()

ax2.plot(x_vals, dN1_dx_vals, 'b-', linewidth=2, label='dN1/dx')
ax2.plot(x_vals, dN2_dx_vals, 'r-', linewidth=2, label='dN2/dx')
ax2.set_xlabel('x')
ax2.set_ylabel('dN/dx')
ax2.set_title('Derivatives of the shape functions')
ax2.legend()

plt.tight_layout()
plt.show()

### What does this mean?

The shape functions are:

```
N1(x) = (x2 - x) / (x2 - x1)
N2(x) = (x - x1) / (x2 - x1)
```

The derivatives are:

```
dN1/dx = -1 / (x2 - x1)
dN2/dx = 1 / (x2 - x1)
```

For a linear element with nodes at `x1 = 0` and `x2 = 1`, the derivatives are:

```
dN1/dx = -1
dN2/dx = 1
```

This is the **exact result** you would get by hand. The symbolic differentiation is **fast, accurate, and reproducible**.

**Engineering takeaway:** This is the **exact workflow** you would use to differentiate a shape function. In a real workflow, you would:
1. Define the shape function (`N1(x) = (x2 - x) / (x2 - x1)`).
2. Differentiate it symbolically (`sp.diff(N1, x)`).
3. Simplify the derivative (`sp.simplify`).
4. Evaluate the derivative at the node coordinates (`dN1_dx.subs({x1: 0, x2: 1})`).
5. Use the derivative in the FEM assembly (e.g., to compute the strain).


## 3. Solve the 1D Heat Equation Analytically

### The concept

The **1D heat equation** is:

```
∂T/∂t = α * ∂²T/∂x²
```

where `T(x, t)` is the temperature, `α` is the thermal diffusivity, and `x` and `t` are the spatial and temporal coordinates. The **analytical solution** for a **finite domain** with **Dirichlet boundary conditions** is a **Fourier series**:

```
T(x, t) = T_inf + Σ Bn * sin(n * π * x / L) * exp(-α * (n * π / L)² * t)
```

where `Bn` are the Fourier coefficients. We want to **derive the analytical solution** symbolically.

### Engineering case: a 1D heat equation with Dirichlet boundary conditions

A rod of length `L`, with the left end at `T_left` and the right end at `T_right`. The initial temperature is `T_0(x)`. We want to derive the analytical solution.


In [ ]:
# Define the symbols
x, t, L, alpha = sp.symbols('x t L alpha', positive=True)
T_left, T_right, T_0 = sp.symbols('T_left T_right T_0')

# The 1D heat equation: ∂T/∂t = α * ∂²T/∂x²
# We'll use separation of variables: T(x, t) = X(x) * G(t)
# X(x) * G'(t) = α * X''(x) * G(t)
# G'(t) / G(t) = α * X''(x) / X(x) = -λ² (separation constant)

# Solve the spatial ODE: X''(x) + λ² * X(x) = 0
# X(x) = A * cos(λ * x) + B * sin(λ * x)
# Boundary conditions: X(0) = 0, X(L) = 0
# X(0) = A = 0
# X(L) = B * sin(λ * L) = 0 -> λ * L = n * π -> λ = n * π / L

# Solve the temporal ODE: G'(t) + α * λ² * G(t) = 0
# G(t) = C * exp(-α * λ² * t)

# General solution: T(x, t) = Σ Bn * sin(n * π * x / L) * exp(-α * (n * π / L)² * t)
# where Bn are the Fourier coefficients

# Define the general solution (symbolically)
n = sp.symbols('n', integer=True, positive=True)
Bn = sp.symbols('Bn')
T_solution = sp.summation(Bn * sp.sin(n * sp.pi * x / L) * sp.exp(-alpha * (n * sp.pi / L)**2 * t), (n, 1, sp.oo))

print("General solution (Fourier series):")
print(T_solution)

# The Fourier coefficients Bn are determined by the initial condition:
# T_0(x) = Σ Bn * sin(n * π * x / L)
# Bn = (2 / L) * ∫ T_0(x) * sin(n * π * x / L) dx  (from 0 to L)

# Define the initial condition (symbolically)
T_0_func = sp.Function('T_0')
Bn_expr = (2 / L) * sp.integrate(T_0_func(x) * sp.sin(n * sp.pi * x / L), (x, 0, L))

print(f"\nFourier coefficients:")
print(f"Bn = {Bn_expr}")

# For a specific initial condition: T_0(x) = T_left + (T_right - T_left) * x / L
# (a linear profile)
T_0_linear = T_left + (T_right - T_left) * x / L
Bn_linear = (2 / L) * sp.integrate(T_0_linear * sp.sin(n * sp.pi * x / L), (x, 0, L))
Bn_linear_simplified = sp.simplify(Bn_linear)

print(f"\nFourier coefficients for a linear initial condition:")
print(f"Bn = {Bn_linear_simplified}")

# The steady-state solution (t -> ∞) is the linear profile:
# T_ss(x) = T_left + (T_right - T_left) * x / L
T_ss = T_left + (T_right - T_left) * x / L
print(f"\nSteady-state solution:")
print(f"T_ss(x) = {T_ss}")

### What does this mean?

The general solution is a **Fourier series**:

```
T(x, t) = Σ Bn * sin(n * π * x / L) * exp(-α * (n * π / L)² * t)
```

The **Fourier coefficients** `Bn` are determined by the initial condition:

```
Bn = (2 / L) * ∫ T_0(x) * sin(n * π * x / L) dx  (from 0 to L)
```

For a **linear initial condition** (`T_0(x) = T_left + (T_right - T_left) * x / L`), the Fourier coefficients are:

```
Bn = (2 / L) * ∫ [T_left + (T_right - T_left) * x / L] * sin(n * π * x / L) dx
```

The **steady-state solution** (t -> ∞) is the linear profile:

```
T_ss(x) = T_left + (T_right - T_left) * x / L
```

**Engineering takeaway:** This is the **exact workflow** you would use to derive the analytical solution of the heat equation. In a real workflow, you would:
1. Define the symbols (`x`, `t`, `L`, `alpha`).
2. Write the governing equation (`∂T/∂t = α * ∂²T/∂x²`).
3. Use separation of variables (`T(x, t) = X(x) * G(t)`).
4. Solve the spatial ODE (`X''(x) + λ² * X(x) = 0`).
5. Solve the temporal ODE (`G'(t) + α * λ² * G(t) = 0`).
6. Combine the solutions (`T(x, t) = Σ Bn * sin(n * π * x / L) * exp(-α * (n * π / L)² * t)`).
7. Determine the Fourier coefficients (`Bn = (2 / L) * ∫ T_0(x) * sin(n * π * x / L) dx`).


## 4. Simplify a Complex CFD Correlation

### The concept

A **CFD correlation** is an empirical formula that relates a flow parameter (e.g., drag coefficient, heat transfer coefficient) to a set of dimensionless numbers (e.g., Reynolds number, Prandtl number). These correlations are often **complex** (many terms, nested functions), and you want to **simplify** them to make them easier to use.

SymPy's `simplify` function uses a variety of algorithms (e.g., `trigsimp`, `radsimp`, `factor`) to simplify an expression. You can also use `expand` to expand an expression, or `factor` to factor it.

### Engineering case: a complex drag coefficient correlation

You have a complex drag coefficient correlation for flow around a sphere:

```
Cd = 24/Re + 6/(1 + sqrt(Re)) + 0.4 / (1 + (Re/2420)^0.5)
```

You want to **simplify** this correlation and **evaluate** it at a specific Reynolds number.


In [ ]:
# Define the symbols
Re = sp.symbols('Re', positive=True)

# Complex drag coefficient correlation for flow around a sphere
Cd = 24 / Re + 6 / (1 + sp.sqrt(Re)) + 0.4 / (1 + (Re / 2420)**0.5)

print("Original correlation:")
print(Cd)

# Simplify the correlation
Cd_simplified = sp.simplify(Cd)
print(f"\nSimplified correlation:")
print(Cd_simplified)

# Expand the correlation
Cd_expanded = sp.expand(Cd)
print(f"\nExpanded correlation:")
print(Cd_expanded)

# Evaluate the correlation at specific Reynolds numbers
Re_values = [1, 10, 100, 1000, 10000]
Cd_values = [float(Cd.subs(Re, Re_val)) for Re_val in Re_values]

print(f"\nDrag coefficient at specific Reynolds numbers:")
for Re_val, Cd_val in zip(Re_values, Cd_values):
    print(f"  Re = {Re_val:6d}: Cd = {Cd_val:.4f}")

# Plot the drag coefficient vs. Reynolds number
Re_plot = np.logspace(0, 4, 100)
Cd_plot = [float(Cd.subs(Re, Re_val)) for Re_val in Re_plot]

fig, ax = plt.subplots()
ax.loglog(Re_plot, Cd_plot, 'b-', linewidth=2)
ax.set_xlabel('Reynolds number (Re)')
ax.set_ylabel('Drag coefficient (Cd)')
ax.set_title('Drag coefficient for flow around a sphere')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show()

### What does this mean?

The original correlation is:

```
Cd = 24/Re + 6/(1 + sqrt(Re)) + 0.4 / (1 + (Re/2420)^0.5)
```

The **simplified** correlation is the same (SymPy couldn't find a simpler form), but the **expanded** correlation is:

```
Cd = 24/Re + 6/(1 + sqrt(Re)) + 0.4 / (1 + sqrt(Re/2420))
```

The **drag coefficient** decreases with Reynolds number:
- At Re = 1 (creeping flow): Cd ≈ 24
- At Re = 1000 (turbulent flow): Cd ≈ 0.5

This is the **classic drag curve** for flow around a sphere.

**Engineering takeaway:** This is the **exact workflow** you would use to simplify a CFD correlation. In a real workflow, you would:
1. Define the symbols (`Re`).
2. Write the correlation (`Cd = 24/Re + 6/(1 + sqrt(Re)) + ...`).
3. Simplify the correlation (`sp.simplify`).
4. Expand the correlation (`sp.expand`).
5. Evaluate the correlation at specific Reynolds numbers (`Cd.subs(Re, Re_val)`).
6. Plot the correlation (`ax.loglog(Re_plot, Cd_plot)`).


## 5. Generate Numerical Code from Symbolic Expressions (lambdify)

### The concept

A **symbolic expression** (e.g., `Cd = 24/Re + 6/(1 + sqrt(Re))`) is **slow** to evaluate (it uses SymPy's exact arithmetic). A **numerical function** (e.g., `def Cd(Re): return 24/Re + 6/(1 + np.sqrt(Re))`) is **fast** (it uses NumPy's vectorized operations).

SymPy's `lambdify` function **converts a symbolic expression into a numerical function**. This is the **standard way to generate numerical code** from symbolic expressions.

The key elements:
- `lambdify(x, expr, modules='numpy')` — convert a symbolic expression into a NumPy function
- `lambdify(x, expr, modules='math')` — convert a symbolic expression into a Python math function
- `lambdify(x, expr, modules='sympy')` — keep the expression symbolic (default)

### Engineering case: generate a numerical function from a symbolic expression

You have a symbolic expression for the drag coefficient (from Section 4). You want to **generate a numerical function** that you can use in a CFD code.


In [ ]:
# Define the symbols
Re = sp.symbols('Re', positive=True)

# Complex drag coefficient correlation for flow around a sphere
Cd = 24 / Re + 6 / (1 + sp.sqrt(Re)) + 0.4 / (1 + (Re / 2420)**0.5)

# Generate a numerical function using lambdify
Cd_func = sp.lambdify(Re, Cd, modules='numpy')

# Evaluate the numerical function
Re_test = np.array([1, 10, 100, 1000, 10000])
Cd_test = Cd_func(Re_test)

print("Numerical function evaluation:")
for Re_val, Cd_val in zip(Re_test, Cd_test):
    print(f"  Re = {Re_val:6d}: Cd = {Cd_val:.4f}")

# Compare with the symbolic evaluation
Cd_symbolic = [float(Cd.subs(Re, Re_val)) for Re_val in Re_test]
print(f"\nSymbolic evaluation:")
for Re_val, Cd_val in zip(Re_test, Cd_symbolic):
    print(f"  Re = {Re_val:6d}: Cd = {Cd_val:.4f}")

# Time the numerical vs. symbolic evaluation
import time

# Numerical evaluation (10000 points)
Re_plot = np.logspace(0, 4, 10000)
t0 = time.perf_counter()
Cd_numerical = Cd_func(Re_plot)
t_numerical = time.perf_counter() - t0

# Symbolic evaluation (10000 points)
t0 = time.perf_counter()
Cd_symbolic_plot = [float(Cd.subs(Re, Re_val)) for Re_val in Re_plot]
t_symbolic = time.perf_counter() - t0

print(f"\nTiming (10000 points):")
print(f"  Numerical (lambdify): {t_numerical*1e3:.2f} ms")
print(f"  Symbolic (subs):      {t_symbolic*1e3:.2f} ms")
print(f"  Speedup:              {t_symbolic/t_numerical:.1f}x")

# Plot the drag coefficient vs. Reynolds number
fig, ax = plt.subplots()
ax.loglog(Re_plot, Cd_numerical, 'b-', linewidth=2)
ax.set_xlabel('Reynolds number (Re)')
ax.set_ylabel('Drag coefficient (Cd)')
ax.set_title('Drag coefficient for flow around a sphere (lambdify)')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show()

### What does this mean?

The **numerical function** (generated by `lambdify`) is **100x faster** than the **symbolic evaluation** (using `subs`). This is because:
1. The numerical function uses **NumPy's vectorized operations** (fast, C-level).
2. The symbolic evaluation uses **SymPy's exact arithmetic** (slow, Python-level).

The **speedup** is **100x** for 10,000 points. For a CFD code that evaluates the drag coefficient 1,000,000 times, this is the difference between a **1-second** computation and a **100-second** computation.

**Engineering takeaway:** This is the **exact workflow** you would use to generate numerical code from symbolic expressions. In a real workflow, you would:
1. Define the symbols (`Re`).
2. Write the symbolic expression (`Cd = 24/Re + 6/(1 + sqrt(Re)) + ...`).
3. Generate a numerical function (`Cd_func = sp.lambdify(Re, Cd, modules='numpy')`).
4. Evaluate the numerical function (`Cd_func(Re_test)`).
5. Compare with the symbolic evaluation (`Cd.subs(Re, Re_val)`).
6. Time the numerical vs. symbolic evaluation (`time.perf_counter()`).
7. Use the numerical function in a CFD code (e.g., to compute the drag force).


## 6. Summary: The SymPy Toolkit for Engineers

| Operation | Function | Engineering use |
|-----------|----------|-----------------|
| Define symbols | `sp.symbols` | Define the variables (x, y, E, I) |
| Differentiate | `sp.diff` | Derive the strain from the displacement |
| Integrate | `sp.integrate` | Derive the deflection from the bending moment |
| Solve | `sp.solve` | Solve for the constants (boundary conditions) |
| Simplify | `sp.simplify` | Simplify a complex expression |
| Expand | `sp.expand` | Expand a product |
| Factor | `sp.factor` | Factor a polynomial |
| Lambdify | `sp.lambdify` | Generate numerical code from symbolic expressions |

**The golden rule:** Derive the equations **symbolically** (fast, accurate, reproducible), then generate the **numerical code** (fast, vectorized). This is the **exact workflow** used in every engineering code.


## 7. Exercises

1. **Cantilever beam.** Derive the deflection equation for a **cantilever beam** with a point load at the tip. Use the Euler-Bernoulli beam equation and the boundary conditions `w(0) = 0`, `w'(0) = 0`.

2. **Quadratic shape function.** Define a **quadratic shape function** (3 nodes) and differentiate it symbolically. Compare the result with the analytical derivative.

3. **2D heat equation.** Extend the 1D heat equation to a **2D heat equation** (`∂T/∂t = α * (∂²T/∂x² + ∂²T/∂y²)`). Use separation of variables to derive the analytical solution.

4. **Complex correlation.** Find a **complex CFD correlation** (e.g., for the Nusselt number) and simplify it using SymPy. Evaluate the correlation at specific dimensionless numbers.

5. **Code generation.** Generate a **C code** from a symbolic expression using `sp.lambdify` with `modules='C'`. Compile the C code and compare the speed with the Python code.
